# 23 SFT: weighted

Each epoch one correct query per question is drawn, cheaper queries more likely (probability proportional to (gold cost / cost)^beta, beta 1).

One epoch of LoRA fine-tuning of Qwen2.5-Coder-7B-Instruct on the train questions (the four held-out validation databases excluded), then greedy predictions for validation, dev and mini-dev with the trained adapter. Scoring happens in `25_sft_evaluate` on a CPU runtime.

**Runtime:** the GPU chosen after the pilot (`20_sft_pilot`). **Resumable:** after a disconnect, run the setup and config cells again, then the cell that was running; training continues from the last checkpoint on Drive (saved every 50 steps).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
EXPORT = "sft_v1"     # training data export (made once by 20_sft_pilot)
R = f"{DRIVE_ROOT}/effsql/runs"
import os
assert os.path.exists(f"{R}/{EXPORT}/export_meta.json"), "run 20_sft_pilot first: it creates the export"
ARM = "weighted"
RUN = f"{EXPORT}_{ARM}_e1"
print(RUN)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Train one epoch (about 8,100 questions, about 508 optimizer steps)

Progress is logged every step (loss, learning rate, tokens per second); the validation loss at steps 250 and 500 and at the end.

In [ ]:
!python -m effsql.train sft --export {EXPORT} --arm {ARM} --run {RUN}

### 2. Predict validation, dev and mini-dev with the adapter (vLLM, about 15 to 25 minutes)

In [ ]:
!python -m effsql.train predict --run {RUN} --splits val,dev,minidev

### Done: disconnect now

*Runtime > Disconnect and delete runtime.* An idle GPU keeps using compute units. When all four methods are done, run `25_sft_evaluate` on a CPU runtime.